**Non linear methods**:This Jupyter notebook calculates non-linear Heart Rate Variability (HRV) metrics using pre-filtered and refined patient $RR$ intervals loaded from a CSV file. For each patient with sufficient data, it computes Detrended Fluctuation Analysis (DFA) to extract short-term ($\alpha_1$) and long-term ($\alpha_2$) scaling exponents, Sample Entropy (SampEn) to evaluate cardiac signal complexity and regularity, and Poincaré plot descriptors ($SD1$, $SD2$, and their ratio) to characterize non-linear geometric variability. Finally, it consolidates these advanced physiological metrics into a structured dataset and saves them into a final output file (final_hrv_non_linear_results.csv) for subsequent health and aging analysis.


File used: refined_rr_signals.csv containing final rr intervals

In [3]:
import pandas as pd
import ast
import pywt
import numpy as np
import csv
from math import floor




def calculate_dfa(rr):
    """
    Python implementation of dfa2.m
    Calculates alpha1 (short-term) and alpha2 (long-term) scaling exponents.
    """
    if len(rr) < 64: # Minimum length for reliable alpha2
        return np.nan, np.nan
        
    y = np.cumsum(rr - np.mean(rr)) # Integrated signal
    L = len(y)
    
    # 1. Box Creation (Logarithmic scale)
    min_box, max_box, box_ratio = 4, L // 4, 2**(1/8)
    scales = []
    n = min_box
    i = 1
    while n <= max_box:
        if n not in scales: scales.append(n)
        n = int(floor(min_box * (box_ratio**i) + 0.5))
        i += 1
    
    fluctuations = []
    for s in scales:
        num_seg = L // s
        # Reshape into segments and calculate RMS of detrended segments
        rms_s = []
        for k in range(num_seg):
            segment = y[k*s : (k+1)*s]
            x = np.arange(s)
            poly = np.polyfit(x, segment, 1) # Linear trend
            trend = np.polyval(poly, x)
            rms_s.append(np.sqrt(np.mean((segment - trend)**2)))
        fluctuations.append(np.mean(rms_s))
    
    # 2. Exponent Calculation (Slopes of log-log plot)
    scales = np.array(scales)
    fluctuations = np.array(fluctuations)
    
    # Alpha 1: 4 to 16 beats
    idx1 = (scales >= 4) & (scales <= 16)
    alpha1 = np.polyfit(np.log10(scales[idx1]), np.log10(fluctuations[idx1]), 1)[0] if any(idx1) else np.nan
    
    # Alpha 2: 16 to 64 beats (per your MATLAB file config)
    idx2 = (scales >= 16) & (scales <= 64)
    alpha2 = np.polyfit(np.log10(scales[idx2]), np.log10(fluctuations[idx2]), 1)[0] if any(idx2) else np.nan
    
    return alpha1, alpha2

def calculate_sampen(L, m=2, r=None):
    """
    Calculates Sample Entropy (SampEn) to assess signal complexity.
    Input:
        L: RR interval time series
        m: Template length
        r: Tolerance level
    """
    if r is None:
        r = 0.2 * np.std(L)
    N = len(L)
    # Split time series and save templates of length m
    xmi = np.array([L[i:i+m] for i in range(N-m)])
    xmj = np.array([L[i:i+m] for i in range(N-m+1)])
    # Sum matches excluding self-matches (B)
    B = np.sum([np.sum(np.abs(xmii - xmj).max(axis=1) <= r) - 1 for xmii in xmi])
    
    # Increase template length to m + 1
    m_new = m + 1
    xm_new = np.array([L[i:i+m_new] for i in range(N-m_new+1)])
    # Sum matches excluding self-matches (A)
    A = np.sum([np.sum(np.abs(xmii_new - xm_new).max(axis=1) <= r) - 1 for xmii_new in xm_new])
    
    return -np.log(A/B) if (A > 0 and B > 0) else np.nan




def poincare_indices(rr):
    """
    Calculates SD1 and SD2 from the Poincaré return map.
    SD1: Short-term variability (minor axis).
    SD2: Long-term variability (major axis).
    """
    diff_rr = np.diff(rr)
    sd1 = np.sqrt(np.var(diff_rr) / 2) #
    sd2 = np.sqrt(2 * np.var(rr) - 0.5 * np.var(diff_rr)) #
    return sd1, sd2



# Configuration
INPUT_FILE = "refined_rr_signals.csv" 
OUTPUT_FILE = "final_hrv_non_linear_results.csv"


# 1. Load your refined data
df = pd.read_csv(INPUT_FILE)

# Convert string-represented lists back to actual Python lists/arrays
df['final_rr_intervals'] = df['final_rr_intervals'].apply(ast.literal_eval)

master_non_linear_results = []

for _, row in df.iterrows():
    pid = row['Patient_ID']
    
    # 2. Use the ALREADY refined intervals from your CSV
    nn_vals = np.array(row['final_rr_intervals'])
    
    # Check for sufficient data length (reliable results require a minimum length)
    if len(nn_vals) > 50: 
        #DFA alpha1 and alpha2
        alpha1, alpha2 = calculate_dfa(nn_vals)

        
        # Sample Entropy (Complexity)
        se = calculate_sampen(nn_vals)
        
        
        # Poincaré Indices (Non-linear Geometric Variability)
        sd1, sd2 = poincare_indices(nn_vals)
        
        # 3. Combine variables for this patient
        combined_nl = {
            "Patient_ID": pid,
            "samp_en": se,
            "dfa_alpha1": alpha1,
            "dfa_alpha2": alpha2,
            "poincare_sd1": sd1,
            "poincare_sd2": sd2,
            "poincare_ratio": sd1/sd2 if sd2 != 0 else np.nan
        }
        master_non_linear_results.append(combined_nl)

# 4. Save to final CSV
final_nl_df = pd.DataFrame(master_non_linear_results)
final_nl_df.to_csv(OUTPUT_FILE, index=False)

print(f"Success! Master non-linear variables saved to {OUTPUT_FILE}.")

C:\Users\34673\AppData\Local\Temp\ipykernel_1284\3985678187.py:56: RankWarning: Polyfit may be poorly conditioned
  alpha2 = np.polyfit(np.log10(scales[idx2]), np.log10(fluctuations[idx2]), 1)[0] if any(idx2) else np.nan


Success! Master non-linear variables saved to final_hrv_non_linear_results.csv.
